# FPL rank cohort exploration

Explore how sampled managers' points and overall ranks change by gameweek. The sampled standings `stratum` is used as the cohort label, so the summaries will compare rank bands when the full sample has been collected. Negative `rank_change` means a better rank; positive means a worse rank.

The current pilot has only one manager history. Treat its outputs as a pipeline preview, not as representative cohort findings.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'analysis':
    ROOT = ROOT.parent
DATA = ROOT / 'data'

history_files = sorted((DATA / 'processed').glob('manager_gameweek_history_*.parquet'), key=lambda p: p.stat().st_mtime)
if not history_files:
    raise FileNotFoundError(f'No processed history Parquet found under {DATA / "processed"}')
history_path = history_files[-1]
season = history_path.stem.removeprefix('manager_gameweek_history_')
standings_path = DATA / 'raw' / f'standings_sample_{season}.parquet'

history = pd.read_parquet(history_path)
standings = pd.read_parquet(standings_path) if standings_path.exists() else pd.DataFrame()
print(f'Season: {season} | history file: {history_path}')
print(f'History rows: {len(history):,} | managers: {history.manager_id.nunique():,}')
print(f'Standings file: {standings_path} | found: {standings_path.exists()}')

## Coverage and cohort labels

Attach the sampled standings stratum to each history row. The rank recorded in the standings snapshot is the baseline rank; `overall_rank` in history changes by gameweek.

In [ ]:
if not standings.empty:
    cohort_columns = [c for c in ['manager_id', 'stratum', 'source_page', 'rank', 'snapshot_utc'] if c in standings.columns]
    cohort_lookup = standings[cohort_columns].drop_duplicates('manager_id').rename(columns={'rank': 'sample_rank'})
    df = history.merge(cohort_lookup, on='manager_id', how='left', validate='many_to_one')
else:
    df = history.copy()
    df['stratum'] = pd.NA

df['event'] = pd.to_numeric(df['event'], errors='coerce').astype('Int64')
df = df.sort_values(['manager_id', 'event']).reset_index(drop=True)
coverage = (df.groupby('event', observed=True)
              .agg(managers=('manager_id', 'nunique'),
                   strata=('stratum', 'nunique'),
                   median_points=('points', 'median'),
                   median_overall_rank=('overall_rank', 'median'))
              .reset_index())
coverage

In [ ]:
if not standings.empty:
    (standings.groupby('stratum', dropna=False)
     .agg(managers=('manager_id', 'nunique'),
          first_sample_rank=('rank', 'min'),
          last_sample_rank=('rank', 'max'),
          sampled_page=('source_page', 'first'))
     .reset_index())
else:
    print('Standings sample is not available; rank strata cannot be attached.')

## Rank movement

`overall_rank` is the manager's overall rank at each gameweek. The difference from the prior observed gameweek is easy to interpret: negative is an improvement, positive is a drop.

In [ ]:
df['previous_rank'] = df.groupby('manager_id')['overall_rank'].shift()
df['rank_change'] = df['overall_rank'] - df['previous_rank']
df['rank_change_pct'] = df['rank_change'] / df['previous_rank'] * 100
df[['manager_id', 'stratum', 'event', 'points', 'overall_rank', 'previous_rank', 'rank_change', 'rank_change_pct']].head(15)

In [ ]:
# Overall rank trajectory for each sampled manager (lower rank is better).
fig, ax = plt.subplots(figsize=(11, 5))
for manager_id, manager in df.groupby('manager_id'):
    label = str(manager_id)
    if 'manager_name' in manager.columns:
        label = str(manager['manager_name'].iloc[0])
    ax.plot(manager['event'], manager['overall_rank'], marker='o', alpha=0.65, label=label)
ax.set(title='Overall rank by gameweek', xlabel='Gameweek', ylabel='Overall rank')
ax.invert_yaxis()
if df['manager_id'].nunique() <= 12:
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

## Cohort summaries

Compare weekly points and rank movement within each sampled stratum. Interpret median rank movement alongside the number of managers with a prior gameweek rank (`rank_change_n`).

In [ ]:
cohort_summary = (df.groupby(['stratum', 'event'], dropna=False, observed=True)
    .agg(managers=('manager_id', 'nunique'),
         median_points=('points', 'median'),
         mean_points=('points', 'mean'),
         median_rank=('overall_rank', 'median'),
         median_rank_change=('rank_change', 'median'),
         rank_change_n=('rank_change', 'count'),
         median_rank_change_pct=('rank_change_pct', 'median'))
    .reset_index())
cohort_summary.head(30)

In [ ]:
# One line per cohort shows whether its typical rank is improving or worsening.
fig, ax = plt.subplots(figsize=(11, 5))
for stratum, cohort in cohort_summary.groupby('stratum', dropna=False):
    ax.plot(cohort['event'], cohort['median_rank_change'], marker='o', label=f'Stratum {stratum}')
ax.axhline(0, color='black', linewidth=1, alpha=0.6)
ax.set(title='Median overall-rank change by cohort and gameweek',
       xlabel='Gameweek', ylabel='Median rank change (negative = improved)')
if cohort_summary['stratum'].nunique(dropna=False) <= 12:
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()